# Anotador TDAH · Flujo C · Por escala (9 llamadas por nota)

Punto intermedio: una llamada **por escala ejecutiva** (9 llamadas por nota), cada una con los 4-9 ítems de esa escala. Contexto acotado como en el flujo B pero con un coste 7 veces menor.

Los ids que el modelo devuelva fuera de la escala consultada se descartan y quedan registrados en la traza (`fuera_de_escala`): es en sí mismo un indicador de obediencia al contexto.

**Común a los tres flujos (A/B/C):** el LLM solo produce ítems `{id, evidencia, justificacion}`. Las escalas afectadas y el nivel de alerta **no los decide el modelo**: se derivan de forma determinista con las reglas del instrumento. La evidencia se verifica automáticamente contra la nota.

**Cómo usarlo:** ajustar la celda de parámetros (un solo paciente) y ejecutar de arriba abajo. Comparar flujos en `comparacion_experimentos.ipynb` con los códigos de experimento.

## 1 · Parámetros

In [ ]:
import datetime as dt
import json
import sqlite3
import time
import unicodedata

import pandas as pd

# --- Parámetros del experimento (lo único que hay que tocar) ---
PACIENTE     = "PAC001"    # un solo paciente por ejecución
NOTAS        = None         # None = todas sus notas; o lista de id_entrada: [10000, 10003]
MUESTRA      = 4            # nº máximo de notas a anotar (None = sin límite)
REPETICIONES = 3            # veces que se anota cada nota
TEMPERATURA  = 0.7
MODELO       = "gemma4:e4b" # en Mercurio: gemma4:26b

# --- Rutas y conexión ---
RUTA_BD     = "datos/anotador.db"
OLLAMA_URL  = "http://127.0.0.1:11002"   # puerto del túnel a Ollama
INSTRUMENTO = "instrumentos/brief2.json"

BACKEND     = "langchain"
FLUJO       = "C"
EXPERIMENTO = f"flujo{FLUJO}-{PACIENTE}-t{TEMPERATURA}-{dt.date.today():%Y%m%d}"
print(f"Código de experimento: {EXPERIMENTO}")

## 2 · Datos: las notas del paciente

In [ ]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

notas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    WHERE e.id_paciente = ?
    ORDER BY e.fecha
    ''',
    con, params=[PACIENTE],
)
if NOTAS:
    notas = notas[notas["id_entrada"].isin(NOTAS)]
if MUESTRA:
    notas = notas.head(MUESTRA)


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


notas["edad"] = [
    calcular_edad(n, f) for n, f in zip(notas["fecha_nacimiento"], notas["fecha"])
]

print(f"{PACIENTE}: {len(notas)} notas seleccionadas")
notas[["id_entrada", "informante", "fecha", "edad", "texto"]].head()

## 3 · Esquema de salida y derivación determinista

El contrato Pydantic (solo ítems con evidencia) y la derivación de escalas y nivel desde `reglas_coherencia`.

In [ ]:
from pydantic import BaseModel, Field


class ItemDetectado(BaseModel):
    id: int = Field(description="Número del ítem del catálogo BRIEF-2")
    evidencia: str = Field(
        description="Cita TEXTUAL de la nota, copiada literalmente, que sustenta el ítem"
    )
    justificacion: str = Field(
        description="Por qué esa evidencia corresponde a este ítem (una frase)"
    )


class ListaItems(BaseModel):
    items: list[ItemDetectado] = Field(default_factory=list)

In [ ]:
# El LLM solo produce ítems. Escalas y nivel de alerta se DERIVAN de forma
# determinista: menos cosas que puede alucinar el modelo, más auditable.
ESCALA_DE_ITEM = {it["id"]: it["escala"] for it in instrumento["items"]}
REGLAS = instrumento["reglas_coherencia"]


def derivar_escalas_y_nivel(ids_items):
    '''Escalas afectadas y nivel de alerta a partir de los ítems detectados.

    Reglas del instrumento (reglas_coherencia de brief2.json):
    alto si nº ítems >= alerta_alto_min_items; moderado si >= alerta_moderado_min_items.
    '''
    unicos = set(ids_items)
    escalas = sorted({ESCALA_DE_ITEM[i] for i in unicos if i in ESCALA_DE_ITEM})
    if len(unicos) >= REGLAS["alerta_alto_min_items"]:
        nivel = "alto"
    elif len(unicos) >= REGLAS["alerta_moderado_min_items"]:
        nivel = "moderado"
    else:
        nivel = "bajo"
    return escalas, nivel


print(derivar_escalas_y_nivel([1, 30, 10, 6]))   # -> 4 ítems = alto
print(derivar_escalas_y_nivel([3]))              # -> 1 ítem  = bajo

## 4 · Prompts

In [ ]:
COMILLAS = '"' * 3

ESCALAS = sorted({it["escala"] for it in instrumento["items"]})
ITEMS_POR_ESCALA = {
    esc: [it for it in instrumento["items"] if it["escala"] == esc]
    for esc in ESCALAS
}


def prompt_sistema_escala(instrumento, escala):
    items_esc = ITEMS_POR_ESCALA[escala]
    catalogo = "\n".join(f"  {it['id']}: {it['texto']}" for it in items_esc)
    descripcion = instrumento["escalas"][escala]
    return f'''Eres un {instrumento["rol_anotador"]}.

Analiza la nota de un cuidador y decide qué ítems de la escala «{escala}»
({descripcion}) del instrumento {instrumento["nombre"]} se observan en ella.

## ÍTEMS DE LA ESCALA ({len(items_esc)})
{catalogo}

## INSTRUCCIONES
Para CADA ítem que detectes aporta "id", "evidencia" (cita TEXTUAL literal de
la nota; si no puedes citar, no incluyas el ítem) y "justificacion" (una frase).
Solo puedes usar los ids de esta escala. Si ninguno se observa, lista vacía.'''


def prompt_usuario_nota(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años · Sexo: {e.sexo} · Informante: {e.informante}

## NOTA DEL CUIDADOR
{COMILLAS}{e.texto}{COMILLAS}

Identifica los ítems de la escala que se observan, con su evidencia.'''


print(f"{len(ESCALAS)} escalas: {ESCALAS}")
print()
print(prompt_sistema_escala(instrumento, ESCALAS[0])[:400] + "\n[...]")

## 5 · El modelo y la función de anotación del flujo

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=MODELO,
    base_url=OLLAMA_URL,
    temperature=TEMPERATURA,
    num_predict=2048,
)

In [ ]:
llm_escala = llm.with_structured_output(ListaItems, method="json_schema")

N_LLAMADAS_POR_NOTA = len(ESCALAS)   # 9


def anotar_nota(e):
    '''9 llamadas: una por escala, cada una con sus 4-9 ítems.

    Devuelve (items, ok_llamadas, traza):
    - items: unión de los ítems detectados por las 9 llamadas
    - ok_llamadas: True si TODAS las llamadas devolvieron salida válida
    - traza: salida por escala; los ids fuera de la escala se descartan y
      quedan registrados en la traza como 'fuera_de_escala'
    '''
    items, traza, ok = [], [], True
    usuario = prompt_usuario_nota(e)
    for escala in ESCALAS:
        ids_validos = {it["id"] for it in ITEMS_POR_ESCALA[escala]}
        try:
            obj = llm_escala.invoke([
                SystemMessage(prompt_sistema_escala(instrumento, escala)),
                HumanMessage(usuario),
            ])
        except Exception as exc:
            traza.append({"escala": escala, "error": str(exc)})
            ok = False
            continue
        if obj is None:
            traza.append({"escala": escala, "error": "salida None"})
            ok = False
            continue
        datos = obj if isinstance(obj, dict) else obj.model_dump()
        dentro = [it for it in datos.get("items", []) if it["id"] in ids_validos]
        fuera = [it["id"] for it in datos.get("items", []) if it["id"] not in ids_validos]
        items.extend(dentro)
        traza.append({"escala": escala,
                      "detectados": [it["id"] for it in dentro],
                      "fuera_de_escala": fuera})
    return items, ok, traza

## 6 · Verificación automática de la evidencia

In [ ]:
def _normalizar(s):
    '''Minúsculas, sin tildes y espacios colapsados.'''
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return " ".join(s.split())


def verificar_evidencias(items, texto_nota):
    '''Fracción de ítems cuya evidencia aparece literalmente en la nota.

    Devuelve (fraccion, detalle) con detalle = [(id_item, True/False), ...].
    (None, []) si no hay ítems.
    '''
    if not items:
        return None, []
    t = _normalizar(texto_nota)
    detalle = [
        (it.get("id"), _normalizar(it.get("evidencia", "")) in t) for it in items
    ]
    return sum(ok for _, ok in detalle) / len(detalle), detalle

## 7 · Una nota de ejemplo

In [ ]:
ejemplo = notas.iloc[0]
print(f"Nota {ejemplo.id_entrada} · {ejemplo.informante} · {ejemplo.fecha}")
print(f"Texto: {ejemplo.texto[:250]}\n")

t0 = time.time()
items, ok_llamadas, traza = anotar_nota(ejemplo)
print(f"Latencia total de la nota: {time.time() - t0:.1f}s "
      f"({N_LLAMADAS_POR_NOTA} llamada(s) al modelo)\n")

fraccion, detalle = verificar_evidencias(items, ejemplo.texto)
verificado = dict(detalle)
escalas, nivel = derivar_escalas_y_nivel([it["id"] for it in items])

print(f"Ítems detectados : {sorted(it['id'] for it in items)}")
print(f"Escalas (derivadas): {escalas}")
print(f"Nivel (derivado)   : {nivel}")
if fraccion is not None:
    print(f"Evidencia verificada: {fraccion:.0%}")
print()
for it in items:
    marca = "OK " if verificado.get(it["id"]) else "?? "
    print(f"  [{marca}] ítem {it['id']}: \"{it['evidencia']}\"")
    print(f"        → {it['justificacion']}")

## 8 · El experimento

Cada fila de `experimento` es la anotación COMPLETA de una nota en una repetición (los flujos B y C agregan sus llamadas en una sola fila; la traza por llamada queda en `respuesta_cruda`). `latencia_s` es el coste total de la nota, comparable entre flujos.

> Relanzar con el mismo código añade filas. Para empezar de cero: `con.execute("DELETE FROM experimento WHERE codigo = ?", [EXPERIMENTO]); con.commit()`

In [ ]:
con.execute('''
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,
    escalas_afectadas TEXT,
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL,
    respuesta_cruda   TEXT,
    items_detalle     TEXT,
    evidencia_ok      REAL
)''')
for col, tipo in [("respuesta_cruda", "TEXT"), ("items_detalle", "TEXT"),
                  ("evidencia_ok", "REAL")]:
    try:
        con.execute(f"ALTER TABLE experimento ADD COLUMN {col} {tipo}")
    except sqlite3.OperationalError:
        pass
con.commit()

total_notas = len(notas) * REPETICIONES
total_llamadas = total_notas * N_LLAMADAS_POR_NOTA
print(f"Experimento '{EXPERIMENTO}': {len(notas)} notas × {REPETICIONES} repeticiones "
      f"= {total_notas} anotaciones · {total_llamadas} llamadas al modelo")

hechas = 0
for _, e in notas.iterrows():
    for rep in range(REPETICIONES):
        t0 = time.time()
        items, ok_llamadas, traza = anotar_nota(e)
        latencia = time.time() - t0
        fraccion, _ = verificar_evidencias(items, e.texto)
        escalas, nivel = derivar_escalas_y_nivel([it["id"] for it in items])
        con.execute(
            "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
            "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
            "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s, "
            "respuesta_cruda, items_detalle, evidencia_ok) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (EXPERIMENTO, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
             MODELO, TEMPERATURA, None, e.id_paciente, int(e.id_entrada), rep,
             int(ok_llamadas), json.dumps(sorted(it["id"] for it in items)),
             json.dumps(escalas), nivel, None, None, latencia,
             json.dumps(traza, ensure_ascii=False),
             json.dumps(items, ensure_ascii=False), fraccion),
        )
        con.commit()
        hechas += 1
        ev = f"evidencia {fraccion:.0%}" if fraccion is not None else "sin ítems"
        print(f"  [{hechas:>3}/{total_notas}] nota {e.id_entrada} rep {rep + 1} → "
              f"{len(items)} ítems · nivel {nivel} · {ev} ({latencia:.1f}s)")

print(f"\nGuardado con codigo = '{EXPERIMENTO}'")

## Resultados

In [ ]:
df = pd.read_sql(
    "SELECT * FROM experimento WHERE codigo = ?", con, params=[EXPERIMENTO]
)
print(f"{len(df)} anotaciones del experimento '{EXPERIMENTO}'\n")


def acuerdo_modal(valores):
    s = valores.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


def jaccard_repeticiones(grupo):
    '''Jaccard medio entre pares de repeticiones (estabilidad de la selección).'''
    conjuntos = [frozenset(json.loads(x)) for x in grupo]
    if len(conjuntos) < 2:
        return None
    pares, total = 0, 0.0
    for i in range(len(conjuntos)):
        for j in range(i + 1, len(conjuntos)):
            a, b = conjuntos[i], conjuntos[j]
            total += len(a & b) / len(a | b) if (a | b) else 1.0
            pares += 1
    return round(total / pares, 2)


resumen = df.groupby("id_entrada").agg(
    repeticiones=("repeticion", "count"),
    formato_ok=("formato_ok", "mean"),
    jaccard_items=("items_detectados", jaccard_repeticiones),
    acuerdo_nivel=("nivel_alerta", acuerdo_modal),
    evidencia_media=("evidencia_ok", "mean"),
    latencia_media=("latencia_s", "mean"),
).round(2)

print(f"Formato válido (todas las llamadas)   : {df['formato_ok'].mean():.0%}")
print(f"Jaccard de ítems entre repeticiones   : {resumen['jaccard_items'].mean():.2f}")
print(f"Acuerdo del nivel (derivado)          : {resumen['acuerdo_nivel'].mean():.2f}")
print(f"Evidencia verificada (media)          : {df['evidencia_ok'].mean():.0%}")
print(f"Latencia media por NOTA               : {df['latencia_s'].mean():.1f}s")
resumen

## 9 · Comparación entre flujos

Ejecutar los tres cuadernos (A, B y C) con **el mismo paciente, las mismas notas, el mismo modelo y la misma temperatura**, y comparar los códigos de experimento en `comparacion_experimentos.ipynb`:

- **Estabilidad**: Jaccard de ítems y acuerdo del nivel entre repeticiones.
- **Fidelidad**: evidencia verificada (¿las citas existen?).
- **Coste**: latencia por nota (A: 1 llamada · C: 9 · B: 63).

La pregunta de la ablación: ¿cuánta estructura de contexto necesita el modelo para anotar de forma estable, y a qué coste?